In [ ]:
!apt-get install -y build-essential cmake git
!pip install huggingface_hub

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
git is already the newest version (1:2.34.1-1ubuntu1.15).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [ ]:
!pip install torch transformers sentencepiece protobuf numpy gguf

In [ ]:
!pip install -r /content/llama.cpp/requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment


In [ ]:
from huggingface_hub import snapshot_download
import os

# If model is gated, set your HF token:
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")

model_id = "Qwen/Qwen2.5-7B-Instruct"
local_dir = "/content/qwen2.5-7b-instruct"

snapshot_download(
    repo_id=model_id,
    local_dir=local_dir,
    ignore_patterns=["*.bin", "*.pt"]  # download safetensors only
)
print("Download complete!")

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Download complete!


In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/qwen2.5-7b-instruct \
    --outfile /content/qwen2.5-7b-instruct-f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: qwen2.5-7b-instruct
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.bfloat16 --> F16, shape = {3584, 152064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.bfloat16 --> F32, shape = {3584}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.bfloat16 --> F16, shape = {18944, 3584}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.bfloat16 --> F16, shape = {3584, 18944}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.bf

In [ ]:
# Find the latest pre-built release
!wget -q https://github.com/ggerganov/llama.cpp/releases/latest -O /tmp/latest.html
!grep -o 'releases/tag/[^"]*' /tmp/latest.html | head -1

releases/tag/*name


In [ ]:
# Download pre-built Ubuntu binaries
!wget https://github.com/ggerganov/llama.cpp/releases/download/b5572/llama-b5572-bin-ubuntu-x64.zip -O /content/llama.zip
!unzip -q /content/llama.zip -d /content/llama-bin
!ls /content/llama-bin  # confirm llama-quantize is there

--2026-03-08 12:55:13--  https://github.com/ggerganov/llama.cpp/releases/download/b5572/llama-b5572-bin-ubuntu-x64.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/ggml-org/llama.cpp/releases/download/b5572/llama-b5572-bin-ubuntu-x64.zip [following]
--2026-03-08 12:55:14--  https://github.com/ggml-org/llama.cpp/releases/download/b5572/llama-b5572-bin-ubuntu-x64.zip
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/612354784/798f0fef-4eb1-4a4d-909b-44577650050f?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-08T13%3A55%3A50Z&rscd=attachment%3B+filename%3Dllama-b5572-bin-ubuntu-x64.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b89

In [ ]:
import requests, json

# Get latest release info
release = requests.get("https://api.github.com/repos/ggerganov/llama.cpp/releases/latest").json()
tag = release["tag_name"]
print(f"Latest tag: {tag}")
print("\nAvailable assets:")
for a in release["assets"]:
    print(a["name"])

Latest tag: b8234

Available assets:
cudart-llama-bin-win-cuda-12.4-x64.zip
cudart-llama-bin-win-cuda-13.1-x64.zip
llama-b8234-bin-310p-openEuler-aarch64.tar.gz
llama-b8234-bin-310p-openEuler-x86.tar.gz
llama-b8234-bin-910b-openEuler-aarch64-aclgraph.tar.gz
llama-b8234-bin-910b-openEuler-x86-aclgraph.tar.gz
llama-b8234-bin-macos-arm64.tar.gz
llama-b8234-bin-macos-x64.tar.gz
llama-b8234-bin-ubuntu-rocm-7.2-x64.tar.gz
llama-b8234-bin-ubuntu-s390x.tar.gz
llama-b8234-bin-ubuntu-vulkan-x64.tar.gz
llama-b8234-bin-ubuntu-x64.tar.gz
llama-b8234-bin-win-cpu-arm64.zip
llama-b8234-bin-win-cpu-x64.zip
llama-b8234-bin-win-cuda-12.4-x64.zip
llama-b8234-bin-win-cuda-13.1-x64.zip
llama-b8234-bin-win-hip-radeon-x64.zip
llama-b8234-bin-win-opencl-adreno-arm64.zip
llama-b8234-bin-win-sycl-x64.zip
llama-b8234-bin-win-vulkan-x64.zip
llama-b8234-xcframework.zip


In [ ]:
!mkdir -p /content/llama-bin
!tar -xzf /content/llama.tar.gz -C /content/llama-bin
!ls /content/llama-bin/

build  llama-b8234


In [ ]:
!find /content/llama-bin -name "llama-quantize"

/content/llama-bin/llama-b8234/llama-quantize
/content/llama-bin/build/bin/llama-quantize


In [ ]:
!/content/llama-bin/llama-b8234/llama-quantize \
    /content/qwen2.5-7b-instruct-f16.gguf \
    /content/qwen2.5-7b-instruct-Q4_K_M.gguf \
    Q4_K_M

main: build = 8234 (213c4a0b8)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/qwen2.5-7b-instruct-f16.gguf' to '/content/qwen2.5-7b-instruct-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 37 key-value pairs and 339 tensors from /content/qwen2.5-7b-instruct-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:                               general.name 

In [ ]:
from google.colab import files
files.download("/content/qwen2.5-7b-instruct-Q4_K_M.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp /content/qwen2.5-7b-instruct-Q4_K_M.gguf "/content/drive/MyDrive/qwen2.5-7b-instruct-Q4_K_M.gguf"
print("Done! File saved to Google Drive.")

Done! File saved to Google Drive.
